# Real-ESRGAN Video Upscaler for Kaggle GPU

A clean Kaggle notebook for **1080p → AI 4K → 1440p60 web delivery**.

### Workflow
- `RealESRGAN_x2plus` performs the expensive 2× AI upscale.
- Two T4 GPUs process the frames in parallel when available.
- A complete **4K AI master** is created.
- The master is then downscaled to a **2560×1440 web version at the original FPS**.
- The 1440p version is intended for the website because it is much easier to play than 4K60 while retaining the AI-upscaled detail.


## 1. Settings

Upload/attach your video to the Kaggle notebook first. The path below is already set for your `scene5.mp4`.


In [2]:
# ===== USER SETTINGS =====

# Change this only if your input video has a different Kaggle path.
INPUT_VIDEO = "/kaggle/input/datasets/dhruvmatliwala/my-video/scene5.mp4"

# 1080p -> 4K is exactly 2x.
SCALE = 2
MODEL = "RealESRGAN_x2plus"

# 4K AI master is kept as an intermediate/master file.
MASTER_VIDEO = "/kaggle/working/upscaled_4K_master.mp4"

# Website-ready output: 1440p at the original FPS.
WEB_WIDTH = 2560
WEB_HEIGHT = 1440
WEB_VIDEO = "/kaggle/working/upscaled_1440p60_web.mp4"

# Final H.264 quality. Lower CRF = larger/higher-quality file.
CRF = 18
PRESET = "slow"

WORK_DIR = "/kaggle/working/realesrgan_video"

print("Input :", INPUT_VIDEO)
print("Model :", MODEL)
print("Scale :", SCALE)
print("Master:", MASTER_VIDEO)
print("Web   :", WEB_VIDEO)


Input : /kaggle/input/datasets/dhruvmatliwala/my-video/scene5.mp4
Scale : 2
Model : RealESRGAN_x2plus
Output: /kaggle/working/upscaled_video.mp4


## 2. Check the Kaggle GPUs


In [3]:
import os
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU detected. In Kaggle select Settings -> Accelerator -> GPU."
    )

GPU_COUNT = torch.cuda.device_count()
print("CUDA GPUs detected:", GPU_COUNT)

for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GB")

subprocess.run(["nvidia-smi"], check=False)


PyTorch: 2.10.0+cu128
CUDA available: True
CUDA GPUs detected: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB
Mon Aug 31 00:27:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|        

CompletedProcess(args=['nvidia-smi'], returncode=0)

## 3. Install Real-ESRGAN


In [4]:
%pip install -q --upgrade pip
%pip install -q basicsr facexlib gfpgan opencv-python-headless

import os
import shutil

REPO = "/kaggle/working/Real-ESRGAN"

if os.path.exists(REPO):
    shutil.rmtree(REPO)

!git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /kaggle/working/Real-ESRGAN

%pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
%pip install -q -e /kaggle/working/Real-ESRGAN

print("Real-ESRGAN installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.9 MB/s eta 0:00:0000:010:01
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Cloning into '/kaggle/working/Real-ESRGAN'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 116 (delta 15), reused 80 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 4.29 MiB | 19.36 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done


## 4. Fix BasicSR / modern torchvision compatibility

Kaggle's newer torchvision versions no longer provide the old
`torchvision.transforms.functional_tensor` import used by some BasicSR releases.
This patch changes it to the current `torchvision.transforms.functional` location.


In [5]:
import os

degradations_file = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

if os.path.exists(degradations_file):
    with open(degradations_file, "r") as f:
        text = f.read()

    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"

    if old in text:
        text = text.replace(old, new)
        with open(degradations_file, "w") as f:
            f.write(text)
        print("BasicSR compatibility patch applied.")
    elif new in text:
        print("BasicSR compatibility patch already present.")
    else:
        print("Expected import was not found; continuing.")
else:
    print("BasicSR degradations.py was not found at the expected path.")


BasicSR compatibility patch applied.


## 5. Download the x2 model


In [6]:
import os
import urllib.request

WEIGHTS_DIR = os.path.join(REPO, "weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, "RealESRGAN_x2plus.pth")
WEIGHTS_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"

if not os.path.exists(WEIGHTS_PATH):
    print("Downloading RealESRGAN_x2plus.pth...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print("Model:", WEIGHTS_PATH)
print("Size:", round(os.path.getsize(WEIGHTS_PATH) / 1024**2, 1), "MB")


Model: /kaggle/working/Real-ESRGAN/weights/RealESRGAN_x2plus.pth
Size: 64.0 MB


## 6. Inspect the input video


In [7]:
import json
import subprocess
import os

if not os.path.isfile(INPUT_VIDEO):
    raise FileNotFoundError(f"Video not found: {INPUT_VIDEO}")

probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,duration",
        "-of", "json",
        INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

info = json.loads(probe.stdout)["streams"][0]

print("Resolution:", info.get("width"), "x", info.get("height"))
print("FPS:", info.get("avg_frame_rate") or info.get("r_frame_rate"))
print("Duration:", info.get("duration"), "seconds")
print("Frames:", info.get("nb_frames", "unknown"))


Resolution: 1920 x 1080
FPS: 60/1
Duration: 3.766667 seconds
Frames: 226


## 7. Extract frames

This creates temporary lossless PNG frames. They are deleted by the optional cleanup cell after the final video is checked.


In [8]:
import os
import subprocess
import shutil

FRAMES_DIR = os.path.join(WORK_DIR, "frames")
UPSCALED_DIR = os.path.join(WORK_DIR, "upscaled")
GPU0_INPUT = os.path.join(WORK_DIR, "gpu0_input")
GPU1_INPUT = os.path.join(WORK_DIR, "gpu1_input")
GPU0_OUTPUT = os.path.join(WORK_DIR, "gpu0_output")
GPU1_OUTPUT = os.path.join(WORK_DIR, "gpu1_output")

# Start clean so rerunning this cell does not mix old frames with new ones.
if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)

for d in [FRAMES_DIR, UPSCALED_DIR, GPU0_INPUT, GPU1_INPUT, GPU0_OUTPUT, GPU1_OUTPUT]:
    os.makedirs(d, exist_ok=True)

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", INPUT_VIDEO,
        "-vsync", "0",
        os.path.join(FRAMES_DIR, "frame%08d.png")
    ],
    check=True
)

frame_files = sorted(
    f for f in os.listdir(FRAMES_DIR)
    if f.lower().endswith(".png")
)

print(f"Extracted {len(frame_files)} frames.")


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Extracted 226 frames.


frame=  226 fps= 17 q=-0.0 Lsize=N/A time=00:00:03.76 bitrate=N/A speed=0.275x    
video:420751kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: unknown


## 8. Split frames between the available T4 GPUs

If Kaggle gives this session two GPUs, the frames are split approximately 50/50.
If only one GPU is available, everything runs on GPU 0.


In [9]:
import os
import shutil

# Clean split/output directories in case this cell is rerun.
for d in [GPU0_INPUT, GPU1_INPUT, GPU0_OUTPUT, GPU1_OUTPUT]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

use_two_gpus = torch.cuda.device_count() >= 2

mid = (len(frame_files) + 1) // 2 if use_two_gpus else len(frame_files)

groups = [
    (frame_files[:mid], GPU0_INPUT),
    (frame_files[mid:], GPU1_INPUT),
]

for files_for_gpu, destination in groups:
    for filename in files_for_gpu:
        src = os.path.join(FRAMES_DIR, filename)
        dst = os.path.join(destination, filename)
        os.symlink(src, dst)

print("Using two GPUs:", use_two_gpus)
print("GPU 0 frames:", len(groups[0][0]))
print("GPU 1 frames:", len(groups[1][0]) if use_two_gpus else 0)


Using two GPUs: True
GPU 0 frames: 113
GPU 1 frames: 113


## 9. AI upscale


In [10]:
import os
import subprocess
import time

def upscale_process(input_dir, output_dir, visible_gpu, label):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(visible_gpu)

    cmd = [
        "python",
        f"{REPO}/inference_realesrgan.py",
        "-n", MODEL,
        "-i", input_dir,
        "-o", output_dir,
        "-s", str(SCALE),
        "--suffix", "out",
        "--tile", "512"
    ]

    print(f"Starting {label} on physical GPU {visible_gpu}...")
    return subprocess.Popen(cmd, env=env)

start = time.time()

p0 = upscale_process(GPU0_INPUT, GPU0_OUTPUT, 0, "GPU 0")

if use_two_gpus:
    p1 = upscale_process(GPU1_INPUT, GPU1_OUTPUT, 1, "GPU 1")
    rc1 = p1.wait()
else:
    p1 = None
    rc1 = 0

rc0 = p0.wait()

if rc0 != 0:
    raise RuntimeError(f"GPU 0 Real-ESRGAN process failed with exit code {rc0}")

if rc1 != 0:
    raise RuntimeError(f"GPU 1 Real-ESRGAN process failed with exit code {rc1}")

elapsed = time.time() - start

print(f"AI upscale finished in {elapsed / 60:.1f} minutes.")


Starting GPU 0 on physical GPU 0...
Starting GPU 1 on physical GPU 1...
Testing 0 frame00000001
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12
Testing 1 frame00000002
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12
Testing 2 frame00000003
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12
Testing 3 frame00000004
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12
Testing 4 frame00000005
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12
Testing 5 frame00000006
	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/1

## 10. Combine the upscaled frames


In [11]:
import os
import shutil

# Combine both GPU output folders into one output folder.
if os.path.isdir(UPSCALED_DIR):
    shutil.rmtree(UPSCALED_DIR)
os.makedirs(UPSCALED_DIR, exist_ok=True)

all_outputs = []

for source_dir in [GPU0_OUTPUT, GPU1_OUTPUT]:
    for filename in os.listdir(source_dir):
        if filename.lower().endswith(".png"):
            src = os.path.join(source_dir, filename)
            dst = os.path.join(UPSCALED_DIR, filename)
            shutil.copy2(src, dst)
            all_outputs.append(filename)

all_outputs.sort()

expected = len(frame_files)
actual = len(all_outputs)

print("Expected frames:", expected)
print("Upscaled frames:", actual)

if actual != expected:
    raise RuntimeError("Frame count mismatch. Do not rebuild the video.")

print("All frames are ready.")


Expected frames: 226
Upscaled frames: 226
All frames are ready.


## 11. Rebuild the 4K AI master

This creates the complete 3840×2160 master from all upscaled frames.
It does **not** use `-shortest`, so the full video frame sequence is preserved.
Audio is copied from the original when available.


In [19]:
import subprocess
import os

fps_result = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)
fps = fps_result.stdout.strip()

frame_pattern = os.path.join(UPSCALED_DIR, "frame%08d_out.png")

cmd = [
    "ffmpeg", "-y",
    "-framerate", fps,
    "-i", frame_pattern,
    "-i", INPUT_VIDEO,
    "-map", "0:v:0",
    "-map", "1:a?",
    "-c:v", "libx264",
    "-crf", str(CRF),
    "-preset", PRESET,
    "-pix_fmt", "yuv420p",
    "-c:a", "copy",
    MASTER_VIDEO
]

print("Rebuilding 4K master...")
subprocess.run(cmd, check=True)

print("Created:", MASTER_VIDEO)
print("Size (MB):", round(os.path.getsize(MASTER_VIDEO) / 1024**2, 2))


Rebuilding full video...


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Created: /kaggle/working/upscaled_video.mp4
Size (MB): 10.68


frame=  226 fps=3.4 q=-1.0 Lsize=   10932kB time=00:00:03.76 bitrate=23808.3kbits/s speed=0.0571x    
video:10837kB audio:88kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.067980%
[libx264 @ 0x567eacf6c280] frame I:2     Avg QP:18.31  size:213910
[libx264 @ 0x567eacf6c280] frame P:57    Avg QP:19.84  size:117239
[libx264 @ 0x567eacf6c280] frame B:167   Avg QP:19.99  size: 23870
[libx264 @ 0x567eacf6c280] consecutive B-frames:  1.3%  0.0%  1.3% 97.3%
[libx264 @ 0x567eacf6c280] mb I  I16..4: 27.1% 55.4% 17.5%
[libx264 @ 0x567eacf6c280] mb P  I16..4:  2.6%  2.9%  0.3%  P16..4: 55.2% 10.9%  6.6%  0.0%  0.0%    skip:21.5%
[libx264 @ 0x567eacf6c280] mb B  I16..4:  0.1%  0.0%  0.0%  B16..8: 29.6%  0.9%  0.1%  direct: 0.9%  skip:68.4%  L0:46.8% L1:51.4% BI: 1.8%
[libx264 @ 0x567eacf6c280] 8x8 transform intra:51.4% inter:90.2%
[libx264 @ 0x567eacf6c280] coded y,uvDC,uvAC intra: 33.6% 40.1% 9.6% inter: 9.8% 16.1% 0.1%
[libx264 @ 0x567eacf6c280] i16 v,h,dc,p: 24% 24%  8% 4

## 12. Create the 1440p60 website version

This uses the already-upscaled 4K master, so **Real-ESRGAN is not run again**.
The 1440p version is much easier for typical devices to decode while retaining
the sharper AI-upscaled appearance.


In [20]:
import subprocess
import os

cmd = [
    "ffmpeg", "-y",
    "-i", MASTER_VIDEO,
    "-vf", f"scale={WEB_WIDTH}:{WEB_HEIGHT}:flags=lanczos",
    "-c:v", "libx264",
    "-profile:v", "main",
    "-level", "5.1",
    "-preset", PRESET,
    "-crf", str(CRF),
    "-pix_fmt", "yuv420p",
    "-c:a", "copy",
    WEB_VIDEO
]

print("Creating website version...")
subprocess.run(cmd, check=True)

print("Created:", WEB_VIDEO)
print("Size (MB):", round(os.path.getsize(WEB_VIDEO) / 1024**2, 2))


Video: 3840 x 2160
Video codec: h264
FPS: 60/1
Audio: aac 44100 Hz
Output size (MB): 10.68


## 13. Verify both outputs


In [21]:
import subprocess
import json
import os

def probe_video(path):
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_streams",
        "-show_format",
        "-of", "json",
        path
    ], capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

for label, path in [("4K master", MASTER_VIDEO), ("1440p web", WEB_VIDEO)]:
    data = probe_video(path)
    video = next(s for s in data["streams"] if s.get("codec_type") == "video")
    print(f"\n{label}:")
    print("  Resolution:", video.get("width"), "x", video.get("height"))
    print("  FPS:", video.get("avg_frame_rate"))
    print("  Frames:", video.get("nb_frames"))
    print("  Duration:", video.get("duration"), "seconds")
    print("  Size:", round(os.path.getsize(path) / 1024**2, 2), "MB")

    audio = [s for s in data["streams"] if s.get("codec_type") == "audio"]
    print("  Audio:", audio[0].get("codec_name") if audio else "none")

print("\nWebsite file:", WEB_VIDEO)


/kaggle/working/upscaled_video.mp4

## 14. Download the website-ready 1440p60 video


In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    WEB_VIDEO,
    result_html_prefix="Download website-ready 1440p60 video: "
))
display(FileLink(
    MASTER_VIDEO,
    result_html_prefix="Download 4K AI master: "
))
